In [3]:
from google.colab import files

uploaded = files.upload()

Saving presolver_milp_handoff.pkl to presolver_milp_handoff.pkl


In [4]:
import pickle

with open("presolver_milp_handoff.pkl", "rb") as f:
    handoff = pickle.load(f)

print("Handoff loaded successfully!")

Handoff loaded successfully!


In [5]:
import numpy as np

c = np.asarray(handoff["objective"], dtype=float)

A = handoff["constraint_matrix"]

if hasattr(A, "toarray"):
    A = A.toarray()

A = np.asarray(A, dtype=float)

variable_names = handoff["variable_names"]
variable_types = handoff["variable_types"]

lower_bounds = np.asarray(
    handoff["lower_bounds"],
    dtype=float
)

upper_bounds = np.asarray(
    handoff["upper_bounds"],
    dtype=float
)

constraint_lower = np.asarray(
    handoff["constraint_lower"],
    dtype=float
)

constraint_upper = np.asarray(
    handoff["constraint_upper"],
    dtype=float
)

integrality = np.asarray(
    handoff["integrality"]
)

print("A shape:", A.shape)
print("c length:", len(c))
print("Number of variables:", len(variable_names))
print("Number of constraints:", A.shape[0])

A shape: (112, 87)
c length: 87
Number of variables: 87
Number of constraints: 112


In [6]:
import time
import numpy as np

def solve_admm(
    c,
    A,
    constraint_lower,
    constraint_upper,
    lower_bounds,
    upper_bounds,
    rho=1.0,
    tolerance=1e-4,
    max_iter=2000
):

    m, n = A.shape

    # Initial values
    x = np.zeros(n)
    z = np.clip(
        A @ x,
        constraint_lower,
        constraint_upper
    )

    u = np.zeros(m)

    # Matrix for x-update
    K = rho * (A.T @ A)

    # Numerical stability
    K = K + 1e-8 * np.eye(n)

    objective_history = []
    primal_history = []
    dual_history = []

    start = time.perf_counter()

    for iteration in range(1, max_iter + 1):

        # X UPDATE
        rhs = rho * A.T @ (z - u) - c

        x = np.linalg.solve(K, rhs)

        # Apply variable bounds
        x = np.clip(
            x,
            lower_bounds,
            upper_bounds
        )

        # Z UPDATE
        z_old = z.copy()

        Ax_plus_u = A @ x + u

        z = np.clip(
            Ax_plus_u,
            constraint_lower,
            constraint_upper
        )

        # DUAL UPDATE
        u = u + A @ x - z

        # Metrics
        objective = c @ x

        primal_residual = np.linalg.norm(
            A @ x - z
        )

        dual_residual = rho * np.linalg.norm(
            A.T @ (z - z_old)
        )

        objective_history.append(objective)
        primal_history.append(primal_residual)
        dual_history.append(dual_residual)

        # Convergence
        if (
            primal_residual < tolerance
            and dual_residual < tolerance
        ):
            break

    runtime = time.perf_counter() - start

    return x, {
        "iterations": iteration,
        "runtime": runtime,
        "objective": objective,
        "primal_residual": primal_residual,
        "dual_residual": dual_residual,
        "objective_history": objective_history,
        "primal_history": primal_history,
        "dual_history": dual_history,
        "converged": (
            primal_residual < tolerance
            and dual_residual < tolerance
        )
    }

In [7]:
x_solution, results = solve_admm(
    c=c,
    A=A,
    constraint_lower=constraint_lower,
    constraint_upper=constraint_upper,
    lower_bounds=lower_bounds,
    upper_bounds=upper_bounds,
    rho=1.0,
    tolerance=1e-4,
    max_iter=2000
)

print("ADMM RESULTS")
print("====================")
print("Converged:", results["converged"])
print("Iterations:", results["iterations"])
print("Runtime:", results["runtime"], "seconds")
print("Objective:", results["objective"])
print("Primal residual:", results["primal_residual"])
print("Dual residual:", results["dual_residual"])

ADMM RESULTS
Converged: False
Iterations: 2000
Runtime: 0.4848005540000031 seconds
Objective: 2919995.5000023944
Primal residual: 13.674794929723207
Dual residual: 2.7985734384722413e-07
